In the abstracts/real.txt file, there are 5 real abstracts taken from Nature articles.

In [ ]:
from openai import OpenAI
import dotenv
import os

import numpy as np
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import seaborn as sns

from jinja2 import Environment, FileSystemLoader, select_autoescape
from typing import Any

from transformers import AutoModel, AutoTokenizer
import torch

# supress every warning
import warnings
warnings.filterwarnings("ignore")

client = OpenAI()

In [ ]:
with open('abstracts/real.txt', "r") as f:
    real_abstracts = f.read().split("\n\n")

print(real_abstracts[0])

Generating fake abstracts

create both a system and a user prompt.

    System prompt The system prompt should direct the model to produce a fake abstract on a topic given by the user.

    User prompt The user prompt should just be a topic.

In [ ]:
def load_template(template_filepath: str, arguments: dict[str, Any]) -> str:
    env = Environment(
        loader=FileSystemLoader(searchpath='./'),
        autoescape=select_autoescape()
    )
    template = env.get_template(template_filepath)
    return template.render(**arguments)

In [ ]:
generation_system_prompt = load_template("prompts/exercise/system.jinja",{})
generation_user_prompt = load_template(
    "prompts/exercise/user.jinja",
    {
        "topic": "Panda foraging in subsaharan Africa, and impact on the local polar bear population",
    }
)

print(generation_system_prompt)
print(generation_user_prompt)

In [ ]:
def chat_response(system_prompt, user_prompt, model, temperature) -> str:
    client = OpenAI()

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=temperature,
        max_tokens=400
    ).choices[0].message.content

    return response

In [ ]:
fake_abstract = chat_response(generation_system_prompt, generation_user_prompt, "gpt-4o-mini", 0.2)


In [ ]:
def extract_topic(abstract):
    extraction_system_prompt = load_template("prompts/exercise/extraction_system.jinja",{})
    extraction_user_prompt = load_template(
        "prompts/exercise/extraction_user.jinja",
        {
            "abstract": abstract,
        }
    )

    response = chat_response(extraction_system_prompt, extraction_user_prompt, "gpt-4o-mini", 0.2)

    return response

topic_sentence = extract_topic(fake_abstract)
topic_sentence

topic_sentences = [extract_topic(abstract) for abstract in real_abstracts]
topic_sentences

def generate_abstract(topic):
    generation_system_prompt = load_template("prompts/exercise/system.jinja",{})
    generation_user_prompt = load_template(
        "prompts/exercise/user.jinja",
        {
            "topic": topic,
        }
    )

    fake_abstract = chat_response(generation_system_prompt, generation_user_prompt, "gpt-4o-mini", 0.2)

    return fake_abstract

In [ ]:
generated_abstracts = [generate_abstract(topic) for topic in topic_sentences]
print(topic_sentences[0])
print("-"*10)
print(generated_abstracts[0])
print("-"*10)
print(real_abstracts[0])


In [ ]:
def get_embedding(texts : list, method = "openai"):
    match method:
        case "openai":
            embeddings = client.embeddings.create(
                model="text-embedding-3-small",
                input=texts
            )
            embeddings = [embedding.embedding for embedding in embeddings.data]

            return np.array(embeddings)
        
        case "bert":
            model_checkpoint = "distilbert/distilbert-base-uncased"

            model = AutoModel.from_pretrained(model_checkpoint)
            tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

            tokens = tokenizer(texts, padding=True, truncation=True, return_tensors="pt")
            with torch.no_grad():
                embeddings = model(**tokens).last_hidden_state[:, 0, :].detach().numpy()

            return embeddings
        
        case _:
            raise ValueError("Invalid method")
        
real_embeddings = get_embedding(real_abstracts)
generated_embeddings = get_embedding(generated_abstracts)
similarity = cosine_similarity(real_embeddings, generated_embeddings)

all_embeddings = np.concatenate([real_embeddings, generated_embeddings])

pca = PCA(n_components=2)
pca.fit(all_embeddings)
real_pca = pca.transform(real_embeddings)
fake_pca = pca.transform(generated_embeddings)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
fig.tight_layout()

# heatmap with names axes and 2sf labels
sns.heatmap(similarity, annot=True, fmt='.2f', xticklabels=False, yticklabels=False, square=False, ax=ax1)
ax1.set_xlabel("Generated Abstracts")
ax1.set_ylabel("Real Abstracts")

ax2.scatter(real_pca[:,0], real_pca[:,1], label="Real")
ax2.scatter(fake_pca[:,0], fake_pca[:,1], label="Fake")
ax2.set_xlabel(f"PCA 1 ({pca.explained_variance_ratio_[0]*100:.2f}%)")
ax2.set_ylabel(f"PCA 2 ({pca.explained_variance_ratio_[1]*100:.2f}%)")
ax2.legend()

plt.show()

In [ ]:
with open('abstracts/real_same.txt', "r") as f:
    real_same_abstracts = f.read().split("\n\n")
same_topic_sentences = [extract_topic(abstract) for abstract in real_same_abstracts]
same_topic_sentences

same_generated_abstracts = [generate_abstract(topic) for topic in same_topic_sentences]

same_real_embeddings = get_embedding(real_same_abstracts)
same_generated_embeddings = get_embedding(same_generated_abstracts)

similarity = cosine_similarity(same_real_embeddings, same_generated_embeddings)

all_embeddings = np.concatenate([same_real_embeddings, same_generated_embeddings])

pca = PCA(n_components=2)
pca.fit(all_embeddings)
real_pca = pca.transform(same_real_embeddings)
fake_pca = pca.transform(same_generated_embeddings)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
fig.tight_layout()

# heatmap with names axes and 2sf labels
sns.heatmap(similarity, annot=True, fmt='.2f', xticklabels=False, yticklabels=False, square=False, ax=ax1)
ax1.set_xlabel("Generated Abstracts")
ax1.set_ylabel("Real Abstracts")

ax2.scatter(real_pca[:,0], real_pca[:,1], label="Real")
ax2.scatter(fake_pca[:,0], fake_pca[:,1], label="Fake")
ax2.set_xlabel(f"PCA 1 ({pca.explained_variance_ratio_[0]*100:.2f}%)")
ax2.set_ylabel(f"PCA 2 ({pca.explained_variance_ratio_[1]*100:.2f}%)")
ax2.legend()

plt.show()